# Fraud Ring Detection Walkthrough

This notebook tells the end-to-end story of the project: generate a realistic synthetic payments ecosystem, turn raw events into user-level graph features, compare tabular and graph-enhanced models, and inspect the highest-risk users like an analyst would.

The core idea is simple: individual accounts can look only mildly suspicious, while the relationships between accounts, devices, IPs, cards, and mule merchants can reveal coordinated fraud rings.

## 1. Project Setup

The notebook is designed to run from the repository root. If the generated artifacts already exist, it reuses them so the walkthrough stays quick for GitHub review, interviews, and demos.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

try:
    from IPython.display import SVG, display
except ImportError:
    SVG = None

    def display(obj):
        print(obj)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src" / "fraud_ring_detection").exists():
            return candidate
    for candidate in [Path.cwd(), Path("/content"), Path("/content/FraudDetection")]:
        candidate = candidate.resolve()
        if (candidate / "src" / "fraud_ring_detection").exists():
            return candidate
    colab_root = Path("/content/FraudDetection")
    if Path("/content").exists() and not colab_root.exists():
        print("Project files not found. Cloning the FraudDetection repository into /content/FraudDetection...")
        subprocess.run(
            ["git", "clone", "https://github.com/charlesclothilde-cmd/FraudDetection.git", str(colab_root)],
            check=True,
        )
        if (colab_root / "src" / "fraud_ring_detection").exists():
            return colab_root.resolve()
    raise FileNotFoundError(
        "Could not find src/fraud_ring_detection. If you are in Colab, run this notebook from the cloned repository "
        "or clone https://github.com/charlesclothilde-cmd/FraudDetection.git first."
    )


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.fraud_ring_detection.data import generate_synthetic_data
from src.fraud_ring_detection.features import build_user_features
from src.fraud_ring_detection.models import train_and_evaluate

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORT_DIR = ROOT / "reports"
MODEL_DIR = ROOT / "models"

ROOT

## 2. Generate the Synthetic Payments Network

The generator creates users, merchants, identities, transactions, known fraud-ring membership, and benign shared-infrastructure groups. The synthetic data intentionally includes innocent sharing patterns such as families, offices, coworking spaces, and student houses so the graph model has to learn more than "shared IP equals fraud."

In [ ]:
required_raw_files = [
    "users.csv",
    "transactions.csv",
    "merchants.csv",
    "user_identity.csv",
    "ring_infrastructure.csv",
    "benign_shared_infrastructure.csv",
]

if not all((RAW_DIR / name).exists() for name in required_raw_files):
    generate_synthetic_data(n_users=8000, n_transactions=30000, n_rings=20, random_state=42)

users = pd.read_csv(RAW_DIR / "users.csv")
transactions = pd.read_csv(RAW_DIR / "transactions.csv", parse_dates=["timestamp"])
merchants = pd.read_csv(RAW_DIR / "merchants.csv")
rings = pd.read_csv(RAW_DIR / "ring_infrastructure.csv")
benign = pd.read_csv(RAW_DIR / "benign_shared_infrastructure.csv")

summary = pd.DataFrame(
    {
        "rows": [len(users), len(transactions), len(merchants), len(rings), len(benign)],
        "description": [
            "users with labels and demographics",
            "payment events with devices, IPs, cards, merchants, and fraud flags",
            "merchant metadata and mule-merchant labels",
            "injected fraud-ring infrastructure assignments",
            "benign shared-infrastructure groups",
        ],
    },
    index=["users", "transactions", "merchants", "ring_infrastructure", "benign_shared_infrastructure"],
)

summary

In [ ]:
label_summary = pd.DataFrame(
    {
        "users": [len(users)],
        "fraud_ring_users": [int(users["is_fraud_ring"].sum())],
        "fraud_ring_rate": [users["is_fraud_ring"].mean()],
        "transaction_fraud_rate": [transactions["is_fraud"].mean()],
        "mule_merchants": [int(merchants["is_mule_merchant"].sum())],
    }
)

ring_mix = (
    users.loc[users["is_fraud_ring"].eq(1), "ring_type"]
    .value_counts()
    .rename_axis("ring_type")
    .reset_index(name="fraud_ring_users")
)

display(label_summary.style.format({"fraud_ring_rate": "{:.2%}", "transaction_fraud_rate": "{:.2%}"}))
display(ring_mix)

## 3. Inspect the Behavior We Want the Model to Learn

Fraud rings are not one generic pattern. This dataset injects account farms, mule-merchant abuse, shared-device rings, and promo abuse. Benign infrastructure sharing is also present, which makes the task closer to a real fraud investigation problem.

In [ ]:
ring_profile = (
    transactions.groupby("ring_type")
    .agg(
        transactions=("transaction_id", "count"),
        users=("user_id", "nunique"),
        fraud_rate=("is_fraud", "mean"),
        online_rate=("is_online", "mean"),
        avg_amount=("amount", "mean"),
        merchants=("merchant_id", "nunique"),
        devices=("device_id", "nunique"),
        ips=("ip_id", "nunique"),
    )
    .sort_values("fraud_rate", ascending=False)
)

ring_profile.style.format({"fraud_rate": "{:.1%}", "online_rate": "{:.1%}", "avg_amount": "${:,.0f}"})

In [ ]:
benign.groupby("benign_share_type").agg(
    groups=("benign_share_group", "count"),
    avg_members=("member_count", "mean"),
    max_members=("member_count", "max"),
).sort_values("groups", ascending=False).style.format({"avg_members": "{:.1f}"})

## 4. Build User-Level Features

The feature builder aggregates transaction behavior and adds relationship-aware signals:

- tabular behavior: transaction volume, amount statistics, online share, night share, merchant diversity
- graph pressure: max and average users sharing devices, IPs, cards, and merchants
- connected components: how large a shared-identifier cluster is
- mule exposure: how often a user touches known mule merchants
- Node2Vec-style embeddings: random-walk co-occurrence vectors over a projected user-user graph

In [ ]:
feature_path = PROCESSED_DIR / "user_features.csv"

if feature_path.exists():
    features = pd.read_csv(feature_path)
else:
    features = build_user_features()

features.shape

In [ ]:
feature_groups = {
    "tabular": ["tx_count", "amount_mean", "online_rate", "night_rate", "merchant_count", "tx_per_merchant"],
    "graph": ["device_id_degree_max", "ip_id_degree_max", "card_id_degree_max", "merchant_degree_max", "component_size", "graph_risk_score"],
    "explainability": ["shared_device_count", "shared_ip_count", "mule_merchant_exposure"],
}

comparison_rows = []
for group_name, columns in feature_groups.items():
    for column in columns:
        comparison_rows.append(
            {
                "feature_group": group_name,
                "feature": column,
                "non_ring_avg": features.loc[features["is_fraud_ring"].eq(0), column].mean(),
                "ring_avg": features.loc[features["is_fraud_ring"].eq(1), column].mean(),
            }
        )

feature_lift = pd.DataFrame(comparison_rows)
feature_lift["ring_to_non_ring_ratio"] = feature_lift["ring_avg"] / feature_lift["non_ring_avg"].replace(0, pd.NA)

feature_lift.sort_values("ring_to_non_ring_ratio", ascending=False).style.format(
    {"non_ring_avg": "{:.2f}", "ring_avg": "{:.2f}", "ring_to_non_ring_ratio": "{:.1f}x"}
)

## 5. Train and Compare Models

The training pipeline compares a flat tabular baseline against graph-enhanced and embedding-enhanced variants. The business metric to watch is `recall_at_top_5pct`: if analysts can only review the riskiest 5% of users, how many fraud-ring users are captured?

In [ ]:
metrics_path = REPORT_DIR / "metrics.csv"
model_path = MODEL_DIR / "fraud_ring_model.joblib"

if not metrics_path.exists() or not model_path.exists():
    train_and_evaluate()

metrics = pd.read_csv(metrics_path).sort_values("average_precision", ascending=False)
metrics.style.format(
    {
        "roc_auc": "{:.3f}",
        "average_precision": "{:.3f}",
        "precision_at_top_5pct": "{:.1%}",
        "recall_at_top_5pct": "{:.1%}",
    }
)

In [ ]:
baseline = metrics.loc[metrics["model"].eq("tabular_logistic")].iloc[0]
best = metrics.iloc[0]
best_top5 = metrics.sort_values("recall_at_top_5pct", ascending=False).iloc[0]

pd.DataFrame(
    {
        "story_point": [
            "Best average-precision model",
            "Average-precision lift over tabular baseline",
            "Best fixed-review-queue model",
            "Recall lift at top 5% over tabular baseline",
        ],
        "value": [
            f"{best['model']} ({best['average_precision']:.3f})",
            f"{best['average_precision'] - baseline['average_precision']:+.3f}",
            f"{best_top5['model']} ({best_top5['recall_at_top_5pct']:.1%})",
            f"{best_top5['recall_at_top_5pct'] - baseline['recall_at_top_5pct']:+.1%}",
        ],
    }
)

## 6. Turn Scores Into an Analyst Queue

A fraud model is only useful if it produces an actionable queue. The training pipeline scores every user with the best model, keeps the top candidates, and carries investigation context such as linked users and linked entities.

In [ ]:
scores_path = PROCESSED_DIR / "user_scores.csv"
top_path = REPORT_DIR / "top_suspicious_users.csv"

scores = pd.read_csv(scores_path)
top_users = pd.read_csv(top_path)

top_users[
    [
        "user_id",
        "risk_score",
        "is_fraud_ring",
        "ring_type",
        "component_size",
        "shared_device_count",
        "shared_ip_count",
        "mule_merchant_exposure",
        "top_linked_users",
        "top_linked_entities",
    ]
].head(10).style.format({"risk_score": "{:.3f}"})

In [ ]:
review_depth = int(len(scores) * 0.05)
review_queue = scores.head(review_depth)

queue_summary = pd.DataFrame(
    {
        "review_depth_users": [review_depth],
        "fraud_ring_users_in_queue": [int(review_queue["is_fraud_ring"].sum())],
        "queue_precision": [review_queue["is_fraud_ring"].mean()],
        "ring_recall": [review_queue["is_fraud_ring"].sum() / scores["is_fraud_ring"].sum()],
    }
)

ring_type_capture = (
    review_queue.loc[review_queue["is_fraud_ring"].eq(1), "ring_type"]
    .value_counts()
    .rename_axis("ring_type")
    .reset_index(name="captured_users")
)

display(queue_summary.style.format({"queue_precision": "{:.1%}", "ring_recall": "{:.1%}"}))
display(ring_type_capture)

## 7. Visualize the Suspicious Component

The dashboard uses the same scored user table and renders a shared-infrastructure component for investigation. Dark outlines mark injected fraud-ring users; warmer colors indicate higher graph risk.

In [ ]:
svg_path = REPORT_DIR / "suspicious_ring.svg"

if svg_path.exists() and SVG is not None:
    display(SVG(filename=str(svg_path)))
elif svg_path.exists():
    print(f"SVG report available at {svg_path}")
else:
    print(f"Missing {svg_path}. Run train_and_evaluate() to create it.")

## 8. What This Project Demonstrates

The baseline tabular model learns useful individual behavior, but the graph-enhanced models add the missing investigation context: users connected through suspicious devices, IPs, cards, merchants, and large shared-infrastructure components.

That is the interview-ready story:

1. Simulate noisy fraud-ring behavior with realistic benign sharing.
2. Build tabular, graph, and Node2Vec-style features at the user level.
3. Compare models with ranking metrics and fixed review-capacity metrics.
4. Convert model scores into an analyst queue with linked-user explanations.
5. Surface the suspicious component in a Streamlit dashboard.

Run the dashboard with:

```bash
python3 -m streamlit run app.py
```